# 🦆 CV Check — WetlandBirds 오리 subset 교차검증

영상 단위 **StratifiedGroupKFold**로 신뢰 가능한 `macro F1 ± std`를 구한다.
single split(0.75)의 불확실성을 걷어내는 게 목적.

- **group = 영상** → 같은 영상 구간이 한 fold에만 (leakage 차단)
- **stratified** → fold별 클래스 비율 유지 (test에 exploring 1개 참사 방지)
- **K-Fold** → 136구간 전부가 정확히 한 번씩 test → 전량 평가

`classifier.py`는 건드리지 않는다. `build_feature_matrix`만 import하고,
groups(영상명)는 라벨 파일 기준으로 재구성한다. agent 런타임과 무관한 진단 전용.

> repo 루트 기준으로 실행. `notebooks/` 에 두고 Jupyter로 열면 됨.

## 1. 설정 — 여기만 바꿔서 재실행

In [7]:
FOLDS = 5              # 데이터 부족으로 5-fold 실패 시 3으로
INCLUDE_STD = False    # True → Feature B (mean+std, 1024-dim) 비교
RANDOM_STATE = 42
CONFIG_PATH = "config.json"

## 2. import & 경로 설정

In [8]:
import json, sys
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder

# repo 루트를 import 경로에 추가 (notebooks/ 에서 열어도 src 가 잡히게)
ROOT = Path.cwd()
if (ROOT / "src").exists():
    root = ROOT
elif (ROOT.parent / "src").exists():
    root = ROOT.parent          # notebooks/ 안에서 실행한 경우
else:
    raise RuntimeError("repo 루트를 찾지 못했습니다. repo 루트나 notebooks/ 에서 실행하세요.")
sys.path.insert(0, str(root))

from src.classifier import build_feature_matrix
print("root:", root)

root: C:\Users\osca0\Github\be-more-agent-duck


## 3. 학습 입력 구성 (라벨 파일 기준)

In [9]:
def load_config(path):
    p = root / path
    if p.exists():
        return json.load(open(p, encoding="utf-8"))
    return {}

config = load_config(CONFIG_PATH)
clip_model     = config.get("clip_model", "ViT-B/32")
frames_per_seg = config.get("frames_per_seg", 10)
boxes_csv      = str(root / config.get("train_boxes_csv", "data/wetlandbirds/duck_segment_boxes.csv"))
train_dir      = root / config.get("train_videos_dir", "data/wetlandbirds/videos")
labels_dir     = root / config.get("train_labels_dir", "data/wetlandbirds")

# load_or_train_classifier와 동일 규칙: 라벨 파일을 기준으로 (영상, 라벨) 쌍 구성
label_files = sorted(labels_dir.glob("labels_*.csv"))
if not label_files:
    raise FileNotFoundError(f"학습 라벨 CSV 없음: {labels_dir} — 어댑터를 먼저 실행하세요.")

video_paths, labels_csvs = [], []
for lc in label_files:
    stem = lc.stem[len("labels_"):]
    vpath = train_dir / f"{stem}.mp4"
    if vpath.exists():
        video_paths.append(str(vpath))
        labels_csvs.append(str(lc))
if not video_paths:
    raise FileNotFoundError("라벨에 대응하는 학습 영상 mp4가 없습니다.")

feat = "mean+std(1024)" if INCLUDE_STD else "mean(512)"
print(f"학습 대상 영상 {len(video_paths)}개, feature={feat}")

학습 대상 영상 36개, feature=mean(512)


## 4. 임베딩 추출 (crop 경로 → 캐시 미사용, 몇 분 소요)

In [10]:
X, y = build_feature_matrix(
    video_paths, labels_csvs, clip_model,
    cache_path=None,                 # crop feature → 캐시 금지
    frames_per_seg=frames_per_seg,
    include_std=INCLUDE_STD,
    boxes_csv=boxes_csv,
)
X = np.asarray(X)
y = np.asarray(y)
print("X:", X.shape, "| 클래스:", {c: int((y==c).sum()) for c in np.unique(y)})

[10:09:44] [classifier] ── 영상 1/36: C:\Users\osca0\Github\be-more-agent-duck\data\wetlandbirds\videos\027-gadwall.mp4
[10:09:44] [classifier] crop 학습 영상 로드: C:\Users\osca0\Github\be-more-agent-duck\data\wetlandbirds\videos\027-gadwall.mp4
[10:09:44]   video_name=027-gadwall, labels=2개
[10:09:45] [classifier] crop 진행: 0/2 구간
[10:09:46] [classifier] ── 영상 2/36: C:\Users\osca0\Github\be-more-agent-duck\data\wetlandbirds\videos\028-gadwall.mp4
[10:09:46] [classifier] crop 학습 영상 로드: C:\Users\osca0\Github\be-more-agent-duck\data\wetlandbirds\videos\028-gadwall.mp4
[10:09:46]   video_name=028-gadwall, labels=3개
[10:09:47] [classifier] crop 진행: 0/3 구간
[10:09:50] [classifier] ── 영상 3/36: C:\Users\osca0\Github\be-more-agent-duck\data\wetlandbirds\videos\029-gadwall.mp4
[10:09:50] [classifier] crop 학습 영상 로드: C:\Users\osca0\Github\be-more-agent-duck\data\wetlandbirds\videos\029-gadwall.mp4
[10:09:50]   video_name=029-gadwall, labels=3개
[10:09:51] [classifier] crop 진행: 0/3 구간
[10:09:53] [classifier

## 5. groups(영상명) 재구성 + 길이 검증

`build_feature_matrix`는 video_paths 순서대로, 영상 내에서는 segment_id 오름차순으로 X를 쌓는다.
각 라벨 파일의 유효 구간 수만큼 영상명을 반복해 groups를 만든다.
crop 실패(min_size 등)로 실제 X가 라벨보다 적으면 어긋나므로 **반드시 길이를 검증**한다.

In [11]:
groups = []
for lc in labels_csvs:
    stem = Path(lc).stem[len("labels_"):]
    df = pd.read_csv(lc, index_col=False)
    df = df[df["label"] != "uncertain"]
    groups.extend([stem] * len(df))
groups = np.array(groups)

if len(groups) != len(X):
    raise RuntimeError(
        f"groups({len(groups)}) != X({len(X)}). crop 실패로 일부 구간이 빠졌습니다. "
        f"이 경우 build_feature_matrix가 groups를 직접 반환하도록 return_groups 확장이 필요합니다."
    )
print(f"groups OK: {len(groups)}구간 / {len(set(groups))}영상")

groups OK: 149구간 / 36영상


## 6. StratifiedGroupKFold 교차검증

In [12]:
le = LabelEncoder()
y_enc = le.fit_transform(y)
classes = list(le.classes_)

n_groups = len(set(groups))
n_splits = min(FOLDS, n_groups)
sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

macro_f1s = []
per_class = defaultdict(list)
empty_class_folds = defaultdict(int)

print(f"=== {n_splits}-Fold CV (group=영상, 총 {len(X)}구간 / {n_groups}영상) ===")
for fold, (tr, te) in enumerate(sgkf.split(X, y_enc, groups), 1):
    clf = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced")
    clf.fit(X[tr], y_enc[tr])
    pred = clf.predict(X[te])

    mf1 = f1_score(y_enc[te], pred, average="macro", zero_division=0)
    macro_f1s.append(mf1)
    for c, f in zip(classes, f1_score(y_enc[te], pred, average=None,
                                      labels=range(len(classes)), zero_division=0)):
        per_class[c].append(f)

    present = set(np.unique(y_enc[te]))
    for ci, c in enumerate(classes):
        if ci not in present:
            empty_class_folds[c] += 1

    te_dist = {c: int((y_enc[te]==i).sum()) for i, c in enumerate(classes)}
    print(f"  fold {fold}: macro F1={mf1:.3f}  test={len(te)}구간/{len(set(groups[te]))}영상  분포={te_dist}")

mean, std = float(np.mean(macro_f1s)), float(np.std(macro_f1s))
print(f"\n▶ macro F1 = {mean:.3f} ± {std:.3f}")
for c in classes:
    v = per_class[c]
    warn = f"  ⚠ {empty_class_folds[c]}개 fold의 test에 없음" if empty_class_folds[c] else ""
    print(f"    {c:10} F1 = {np.mean(v):.3f} ± {np.std(v):.3f}{warn}")

=== 5-Fold CV (group=영상, 총 149구간 / 36영상) ===
  fold 1: macro F1=0.516  test=28구간/8영상  분포={np.str_('exploring'): 13, np.str_('feeding'): 9, np.str_('resting'): 6}
  fold 2: macro F1=0.657  test=30구간/7영상  분포={np.str_('exploring'): 13, np.str_('feeding'): 9, np.str_('resting'): 8}
  fold 3: macro F1=0.633  test=29구간/6영상  분포={np.str_('exploring'): 12, np.str_('feeding'): 8, np.str_('resting'): 9}
  fold 4: macro F1=0.705  test=34구간/7영상  분포={np.str_('exploring'): 16, np.str_('feeding'): 10, np.str_('resting'): 8}
  fold 5: macro F1=0.824  test=28구간/8영상  분포={np.str_('exploring'): 13, np.str_('feeding'): 8, np.str_('resting'): 7}

▶ macro F1 = 0.667 ± 0.100
    exploring  F1 = 0.659 ± 0.103
    feeding    F1 = 0.753 ± 0.106
    resting    F1 = 0.589 ± 0.304


## 해석

- **std가 크면(≳0.08)** — 데이터가 작아 결과가 출렁인다 → **데이터 추가가 답**. detection+새영상은 아직 이르다.
- **std가 작으면(≲0.04)** — 그 평균은 신뢰할 만하다 → 분류기 확정. **detection+held-out 착수 초록불.**
- **⚠ 표시된 클래스** — 일부 fold의 test에 부재 → 그 클래스 F1은 특히 불안정 (resting이 소수 영상에 몰린 경우).

`INCLUDE_STD = True`로 재실행하면 Feature B(mean+std)와 같은 CV로 바로 비교할 수 있다.